# TiRex2 GiftEval Evaluation

This notebook shows how to run [TiRex-2](https://github.com/NX-AI/tirex-2.git) on the gift-eval benchmark.

Make sure you download the gift-eval benchmark and set the `GIFT_EVAL_STORE` variable correctly before running this notebook.

## Setup

Before proceeding, ensure you have the following: (Note: You need a Nvidia GPU with [CUDA compute capabality >= 8.0](https://developer.nvidia.com/cuda-gpus))

The environment is managed by [Pixi](https://pixi.prefix.dev/latest/), apart from the python dependencies it also installs the system-level (CUDA) dependencies (like the CUDA-Toolkit). So setting up the environment through Pixi is highly recommended but not required. See [system requirements](#system-requirements) for more information about system-level dependencies. 

1. **Option 1: Setting up the environment with Pixi**
```bash
curl -fsSL https://pixi.sh/install.sh | sh  # install Pixi if you do not have it already
git clone https://github.com/NX-AI/tirex-2.git && cd tirex-2
pixi install -e example-cu128  # if cuda 12.8 is not available use `example-cu126`
eval "$(pixi shell-hook -e example-cu128)"
```

2. **Option 2: Directly installing TiRex-2 through pip**
```bash
pip install "tirex-2[examples,gluonts,fev]"  # Install TiRex-2 from PyPI with the extras needed for gifteval
```
Make sure that the [system-level dependencies]() are available in the environment to be able to evaluate on GPU. 

## Configuration

Set `GIFT_EVAL_STORE` to the directory created by the Hugging Face download command.


In [ ]:
from pathlib import Path
import os

GIFT_EVAL_STORE = Path('/path/to/gifteval_storage').expanduser()
MODEL_ID = 'NX-AI/TiRex-2-gifteval-zs'  # or: 'NX-AI/TiRex-2-gifteval-pretrain'
RESULTS_CSV = Path('./gifteval_results.csv')
MAX_TASKS = None  # set to an integer for a quick smoke run

if not GIFT_EVAL_STORE.is_dir():
    raise FileNotFoundError(f'GiftEval store not found: {GIFT_EVAL_STORE}')

os.environ['GIFT_EVAL'] = str(GIFT_EVAL_STORE.resolve())


## GiftEval Data Loader

Adapted from GiftEval's data loader with the local NumPy/Pandas frequency alias fixes used by this repo.


In [ ]:
# Copyright (c) 2023, Salesforce, Inc.
# SPDX-License-Identifier: Apache-2
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import math
import os
from collections.abc import Iterable, Iterator
from enum import Enum
from functools import cached_property
from pathlib import Path

import datasets
import pyarrow.compute as pc
from gluonts.dataset import DataEntry
from gluonts.dataset.common import ProcessDataEntry
from gluonts.dataset.split import TestData, TrainingDataset, split
from gluonts.itertools import Map
from gluonts.time_feature import norm_freq_str
from gluonts.transform import Transformation
from pandas.tseries.frequencies import to_offset
from toolz import compose

TEST_SPLIT = 0.1
MAX_WINDOW = 20

M4_PRED_LENGTH_MAP = {
    "A": 6,
    "Q": 8,
    "M": 18,
    "W": 13,
    "D": 14,
    "H": 48,
    # new version fix:
    "h": 48,
    "Y": 6,
}

PRED_LENGTH_MAP = {
    "M": 12,
    "W": 8,
    "D": 30,
    "H": 48,
    "T": 48,
    "S": 60,
    # new version fix:
    "h": 48,
    "s": 60,
    "min": 48,
}

TFB_PRED_LENGTH_MAP = {
    "A": 6,
    "H": 48,
    "Q": 8,
    "D": 14,
    "M": 18,
    "W": 13,
    "U": 8,
    "T": 8,
    # new version fix:
    "min": 8,
    "us": 8,
    "Y": 6,
    "h": 48,
}


class Term(Enum):
    SHORT = "short"
    MEDIUM = "medium"
    LONG = "long"

    @property
    def multiplier(self) -> int:
        if self == Term.SHORT:
            return 1
        elif self == Term.MEDIUM:
            return 10
        elif self == Term.LONG:
            return 15


def itemize_start(data_entry: DataEntry) -> DataEntry:
    data_entry["start"] = data_entry["start"].item()
    return data_entry


class MultivariateToUnivariate(Transformation):
    def __init__(self, field):
        self.field = field

    def __call__(self, data_it: Iterable[DataEntry], is_train: bool = False) -> Iterator:
        for data_entry in data_it:
            item_id = data_entry["item_id"]
            val_ls = list(data_entry[self.field])
            for id, val in enumerate(val_ls):
                univariate_entry = data_entry.copy()
                univariate_entry[self.field] = val
                univariate_entry["item_id"] = item_id + "_dim" + str(id)
                yield univariate_entry


class Dataset:
    def __init__(
        self,
        name: str,
        term: Term | str = Term.SHORT,
        to_univariate: bool = False,
        storage_env_var: str = "GIFT_EVAL",
    ):
        storage_path = Path(os.getenv(storage_env_var))
        self.hf_dataset = datasets.load_from_disk(str(storage_path / name)).with_format("numpy")
        process = ProcessDataEntry(
            self.freq,
            one_dim_target=self.target_dim == 1,
        )

        self.gluonts_dataset = Map(compose(process, itemize_start), self.hf_dataset)
        if to_univariate:
            self.gluonts_dataset = MultivariateToUnivariate("target").apply(self.gluonts_dataset)

        self.term = Term(term)
        self.name = name

    @cached_property
    def prediction_length(self) -> int:
        freq = norm_freq_str(to_offset(self.freq).name)
        if freq.endswith("E"):
            freq = freq[:-1]
        pred_len = M4_PRED_LENGTH_MAP[freq] if "m4" in self.name else PRED_LENGTH_MAP[freq]
        return self.term.multiplier * pred_len

    @cached_property
    def freq(self) -> str:
        return self.hf_dataset[0]["freq"]

    @cached_property
    def target_dim(self) -> int:
        return target.shape[0] if len((target := self.hf_dataset[0]["target"]).shape) > 1 else 1

    @cached_property
    def past_feat_dynamic_real_dim(self) -> int:
        if "past_feat_dynamic_real" not in self.hf_dataset[0]:
            return 0
        elif len((past_feat_dynamic_real := self.hf_dataset[0]["past_feat_dynamic_real"]).shape) > 1:
            return past_feat_dynamic_real.shape[0]
        else:
            return 1

    @cached_property
    def windows(self) -> int:
        if "m4" in self.name:
            return 1
        w = math.ceil(TEST_SPLIT * self._min_series_length / self.prediction_length)
        return min(max(1, w), MAX_WINDOW)

    @cached_property
    def _min_series_length(self) -> int:
        if self.hf_dataset[0]["target"].ndim > 1:
            lengths = pc.list_value_length(pc.list_flatten(pc.list_slice(self.hf_dataset.data.column("target"), 0, 1)))
        else:
            lengths = pc.list_value_length(self.hf_dataset.data.column("target"))
        return min(lengths.to_numpy())

    @cached_property
    def sum_series_length(self) -> int:
        if self.hf_dataset[0]["target"].ndim > 1:
            lengths = pc.list_value_length(pc.list_flatten(self.hf_dataset.data.column("target")))
        else:
            lengths = pc.list_value_length(self.hf_dataset.data.column("target"))
        return sum(lengths.to_numpy())

    @property
    def training_dataset(self) -> TrainingDataset:
        training_dataset, _ = split(self.gluonts_dataset, offset=-self.prediction_length * (self.windows + 1))
        return training_dataset

    @property
    def validation_dataset(self) -> TrainingDataset:
        validation_dataset, _ = split(self.gluonts_dataset, offset=-self.prediction_length * self.windows)
        return validation_dataset

    @property
    def test_data(self) -> TestData:
        _, test_template = split(self.gluonts_dataset, offset=-self.prediction_length * self.windows)
        test_data = test_template.generate_instances(
            prediction_length=self.prediction_length,
            windows=self.windows,
            distance=self.prediction_length,
        )
        return test_data


## Evaluation Glue

This cell defines the GiftEval task iterator, metrics, evaluator, and GluonTS adapter inline.


In [ ]:
import logging
from dataclasses import dataclass
from typing import Any

import pandas as pd
from gluonts.ev.metrics import (
    MAE,
    MAPE,
    MASE,
    MSE,
    MSIS,
    ND,
    NRMSE,
    RMSE,
    SMAPE,
    MeanWeightedSumQuantileLoss,
)
from gluonts.model import evaluate_model
from gluonts.time_feature import get_seasonality


class WarningFilter(logging.Filter):
    def __init__(self, text_to_filter):
        super().__init__()
        self.text_to_filter = text_to_filter

    def filter(self, record):
        return self.text_to_filter not in record.getMessage()


gts_logger = logging.getLogger('gluonts.model.forecast')
gts_logger.addFilter(WarningFilter('The mean prediction is not stored in the forecast data'))

SHORT_DATA = 'm4_yearly m4_quarterly m4_monthly m4_weekly m4_daily m4_hourly electricity/15T electricity/H electricity/D electricity/W solar/10T solar/H solar/D solar/W hospital covid_deaths us_births/D us_births/M us_births/W saugeenday/D saugeenday/M saugeenday/W temperature_rain_with_missing kdd_cup_2018_with_missing/H kdd_cup_2018_with_missing/D car_parts_with_missing restaurant hierarchical_sales/D hierarchical_sales/W LOOP_SEATTLE/5T LOOP_SEATTLE/H LOOP_SEATTLE/D SZ_TAXI/15T SZ_TAXI/H M_DENSE/H M_DENSE/D ett1/15T ett1/H ett1/D ett1/W ett2/15T ett2/H ett2/D ett2/W jena_weather/10T jena_weather/H jena_weather/D bitbrains_fast_storage/5T bitbrains_fast_storage/H bitbrains_rnd/5T bitbrains_rnd/H bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H'
MED_LONG_DATA = 'electricity/15T electricity/H solar/10T solar/H kdd_cup_2018_with_missing/H LOOP_SEATTLE/5T LOOP_SEATTLE/H SZ_TAXI/15T M_DENSE/H ett1/15T ett1/H ett2/15T ett2/H jena_weather/10T jena_weather/H bitbrains_fast_storage/5T bitbrains_rnd/5T bizitobs_application bizitobs_service bizitobs_l2c/5T bizitobs_l2c/H'
PRETTY_NAMES = {
    'saugeenday': 'saugeen',
    'temperature_rain_with_missing': 'temperature_rain',
    'kdd_cup_2018_with_missing': 'kdd_cup_2018',
    'car_parts_with_missing': 'car_parts',
}
ALL_DATASETS = list(dict.fromkeys(SHORT_DATA.split() + MED_LONG_DATA.split()))
DATASET_PROPERTIES = {
    "m4_yearly": {
        "domain": "Econ/Fin",
        "frequency": "A",
        "num_variates": 1
    },
    "m4_quarterly": {
        "domain": "Econ/Fin",
        "frequency": "Q",
        "num_variates": 1
    },
    "m4_monthly": {
        "domain": "Econ/Fin",
        "frequency": "M",
        "num_variates": 1
    },
    "m4_weekly": {
        "domain": "Econ/Fin",
        "frequency": "W",
        "num_variates": 1
    },
    "m4_daily": {
        "domain": "Econ/Fin",
        "frequency": "D",
        "num_variates": 1
    },
    "m4_hourly": {
        "domain": "Econ/Fin",
        "frequency": "H",
        "num_variates": 1
    },
    "electricity": {
        "domain": "Energy",
        "frequency": "W",
        "num_variates": 1
    },
    "ett1": {
        "domain": "Energy",
        "frequency": "W",
        "num_variates": 7
    },
    "ett2": {
        "domain": "Energy",
        "frequency": "W",
        "num_variates": 7
    },
    "solar": {
        "domain": "Energy",
        "frequency": "W",
        "num_variates": 1
    },
    "hospital": {
        "domain": "Healthcare",
        "frequency": "M",
        "num_variates": 1
    },
    "covid_deaths": {
        "domain": "Healthcare",
        "frequency": "D",
        "num_variates": 1
    },
    "us_births": {
        "domain": "Healthcare",
        "frequency": "M",
        "num_variates": 1
    },
    "saugeen": {
        "domain": "Nature",
        "frequency": "M",
        "num_variates": 1
    },
    "temperature_rain": {
        "domain": "Nature",
        "frequency": "D",
        "num_variates": 1
    },
    "kdd_cup_2018": {
        "domain": "Nature",
        "frequency": "D",
        "num_variates": 1
    },
    "jena_weather": {
        "domain": "Nature",
        "frequency": "D",
        "num_variates": 21
    },
    "car_parts": {
        "domain": "Sales",
        "frequency": "M",
        "num_variates": 1
    },
    "restaurant": {
        "domain": "Sales",
        "frequency": "D",
        "num_variates": 1
    },
    "hierarchical_sales": {
        "domain": "Sales",
        "frequency": "W-WED",
        "num_variates": 1
    },
    "loop_seattle": {
        "domain": "Transport",
        "frequency": "D",
        "num_variates": 1
    },
    "sz_taxi": {
        "domain": "Transport",
        "frequency": "H",
        "num_variates": 1
    },
    "m_dense": {
        "domain": "Transport",
        "frequency": "D",
        "num_variates": 1
    },
    "bitbrains_fast_storage": {
        "domain": "Web/CloudOps",
        "frequency": "H",
        "num_variates": 2
    },
    "bitbrains_rnd": {
        "domain": "Web/CloudOps",
        "frequency": "H",
        "num_variates": 2
    },
    "bizitobs_application": {
        "domain": "Web/CloudOps",
        "frequency": "10S",
        "num_variates": 2
    },
    "bizitobs_service": {
        "domain": "Web/CloudOps",
        "frequency": "10S",
        "num_variates": 2
    },
    "bizitobs_l2c": {
        "domain": "Web/CloudOps",
        "frequency": "H",
        "num_variates": 7
    }
}

METRICS = [
    MSE(forecast_type='mean'),
    MSE(forecast_type=0.5),
    MAE(),
    MASE(),
    MAPE(),
    SMAPE(),
    MSIS(),
    RMSE(),
    NRMSE(),
    ND(),
    MeanWeightedSumQuantileLoss(quantile_levels=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]),
]


def gift_eval_dataset_iter():
    for ds_name in ALL_DATASETS:
        for term in ('short', 'medium', 'long'):
            if term in ('medium', 'long') and ds_name not in MED_LONG_DATA.split():
                continue

            if '/' in ds_name:
                ds_key, ds_freq = ds_name.split('/')
                ds_key = PRETTY_NAMES.get(ds_key.lower(), ds_key.lower())
            else:
                ds_key = PRETTY_NAMES.get(ds_name.lower(), ds_name.lower())
                ds_freq = DATASET_PROPERTIES[ds_key]['frequency']

            yield {
                'ds_name': ds_name,
                'ds_key': ds_key,
                'ds_freq': ds_freq,
                'term': term,
            }


def evaluate_dataset(predictor, ds_name, ds_key, ds_freq, term):
    print(f'Processing dataset: {ds_name}')
    ds_config = f'{ds_key}/{ds_freq}/{term}'

    raw_dataset = Dataset(name=ds_name, term=term, to_univariate=False)
    dataset = raw_dataset if raw_dataset.target_dim == 1 else Dataset(name=ds_name, term=term, to_univariate=True)

    predictor.set_prediction_len(dataset.prediction_length)
    predictor.set_ds_freq(ds_freq)
    season_length = get_seasonality(dataset.freq)

    res = evaluate_model(
        predictor,
        test_data=dataset.test_data,
        metrics=METRICS,
        batch_size=1024,
        axis=None,
        mask_invalid_label=True,
        allow_nan_forecast=False,
        seasonality=season_length,
    )
    return {
        'dataset': ds_config,
        'model': predictor.model_id,
        'eval_metrics/MSE[mean]': res['MSE[mean]'].iloc[0],
        'eval_metrics/MSE[0.5]': res['MSE[0.5]'].iloc[0],
        'eval_metrics/MAE[0.5]': res['MAE[0.5]'].iloc[0],
        'eval_metrics/MASE[0.5]': res['MASE[0.5]'].iloc[0],
        'eval_metrics/MAPE[0.5]': res['MAPE[0.5]'].iloc[0],
        'eval_metrics/sMAPE[0.5]': res['sMAPE[0.5]'].iloc[0],
        'eval_metrics/MSIS': res['MSIS'].iloc[0],
        'eval_metrics/RMSE[mean]': res['RMSE[mean]'].iloc[0],
        'eval_metrics/NRMSE[mean]': res['NRMSE[mean]'].iloc[0],
        'eval_metrics/ND[0.5]': res['ND[0.5]'].iloc[0],
        'eval_metrics/mean_weighted_sum_quantile_loss': res['mean_weighted_sum_quantile_loss'].iloc[0],
        'domain': DATASET_PROPERTIES[ds_key]['domain'],
        'num_variates': DATASET_PROPERTIES[ds_key]['num_variates'],
    }


@dataclass
class TiRexGiftEvalWrapper:
    model: Any
    freq: str | None = None
    pred_len: int = 32
    batch_size: int = 512

    def set_ds_freq(self, freq):
        self.freq = freq

    def set_prediction_len(self, pred_len):
        self.pred_len = pred_len

    def predict(self, test_data_input):
        return self.model.forecast_gluon(
            test_data_input,
            prediction_length=self.pred_len,
            output_type='gluonts',
            batch_size=self.batch_size,
        )

    @property
    def model_id(self):
        return 'TiRex2'


## Load TiRex2

The model is loaded with `device="cuda"`.


In [ ]:
from tirex2 import ForecastModel, load_model

model: ForecastModel = load_model(MODEL_ID, device='cuda')


## Run Evaluation

Set `MAX_TASKS` above for a quick smoke run, or leave it as `None` for the full benchmark.


In [ ]:
wrapped_model = TiRexGiftEvalWrapper(model)
results = []

for task_idx, task in enumerate(gift_eval_dataset_iter(), start=1):
    if MAX_TASKS is not None and task_idx > MAX_TASKS:
        break

    task_result = evaluate_dataset(wrapped_model, **task)
    results.append(task_result)
    print(task_result)

results = pd.DataFrame(results)
results.to_csv(RESULTS_CSV, index=False)
print(f'Wrote {len(results)} rows to {RESULTS_CSV}')
results
